In [117]:
import os
import kagglehub
import pandas as pd
import seaborn as sns

# Downloads once, then reuses the cached copy
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(path)
os.listdir(path)

C:\Users\ferna\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2


['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [118]:
# Load every CSV into a dict: dfs["orders"], dfs["customers"], ...
dfs = {
    f.removeprefix("olist_").removesuffix("_dataset.csv").removesuffix(".csv"): pd.read_csv(os.path.join(path, f))
    for f in os.listdir(path)
    if f.endswith(".csv")
}
{name: df.shape for name, df in dfs.items()}

{'customers': (99441, 5),
 'geolocation': (1000163, 5),
 'orders': (99441, 8),
 'order_items': (112650, 7),
 'order_payments': (103886, 5),
 'order_reviews': (99224, 7),
 'products': (32951, 9),
 'sellers': (3095, 4),
 'product_category_name_translation': (71, 2)}

In [119]:
orders, order_items, items, payments, reviews = dfs["orders"], dfs["order_items"],dfs["order_items"], dfs["order_payments"], dfs["order_reviews"]


In [120]:
summary = pd.DataFrame({
    name: {
        "rows": df.shape[0],
        "cols": df.shape[1],
        "cols_with_nulls": int(df.isna().any().sum()),
        "null_cells_%": df.isna().sum().sum() / df.size * 100,
        "duplicate_rows": int(df.duplicated().sum()),
    }
    for name, df in dfs.items()
}).T
summary

,rows,cols,cols_with_nulls,null_cells_%,duplicate_rows
customers,99441.0,5.0,0.0,0.000000,0.0
geolocation,1000163.0,5.0,0.0,0.000000,261831.0
orders,99441.0,8.0,3.0,0.616949,0.0
order_items,112650.0,7.0,0.0,0.000000,0.0
order_payments,103886.0,5.0,0.0,0.000000,0.0
order_reviews,99224.0,7.0,2.0,21.006295,0.0
products,32951.0,9.0,8.0,0.825468,0.0
sellers,3095.0,4.0,0.0,0.000000,0.0
product_category_name_translation,71.0,2.0,0.0,0.000000,0.0


In [121]:
df =(dfs['order_items']
 .merge(dfs['sellers'], on='seller_id', how='left')
 .merge(dfs['orders'], on='order_id', how='left')
 .merge(dfs['products'], on='product_id', how='left')
 .merge(dfs['customers'], on='customer_id', how='left'))

In [122]:
df.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id',
       'order_status', 'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state'],
      dtype='str')

In [123]:
print("Orders without items:", (~orders.order_id.isin(items.order_id)).sum())
print("Orders without payments:", (~orders.order_id.isin(payments.order_id)).sum())
print("Orders without reviews:", (~orders.order_id.isin(reviews.order_id)).sum())
orders[~orders.order_id.isin(items.order_id)].order_status.value_counts()

Orders without items: 775
Orders without payments: 1
Orders without reviews: 768


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [124]:
dfs['order_items'][dfs['order_items']['order_id'] == "001ab0a7578dd66cd4b0a71f5b6e1e41"]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
42,001ab0a7578dd66cd4b0a71f5b6e1e41,1,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
43,001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63


In [125]:
df = df.dropna()

In [126]:
df.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id',
       'order_status', 'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state'],
      dtype='str')

In [127]:
df.sort_values('order_id')

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,598.0,4.0,650.0,28.0,9.0,14.0,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,239.0,2.0,30000.0,50.0,30.0,40.0,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,695.0,2.0,3050.0,33.0,13.0,33.0,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,480.0,1.0,200.0,16.0,10.0,15.0,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,409.0,1.0,3750.0,35.0,40.0,30.0,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,88303,itajai,SC,...,1002.0,3.0,10150.0,89.0,15.0,40.0,0c9aeda10a71f369396d0c04dce13a64,65077,sao luis,MA
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,1206,sao paulo,SP,...,232.0,1.0,8950.0,45.0,26.0,38.0,0da9fe112eae0c74d3ba1fe16de0988b,81690,curitiba,PR
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,80610,curitiba,PR,...,869.0,1.0,967.0,21.0,24.0,19.0,cd79b407828f02fdbba457111c38e4c4,4039,sao paulo,SP
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,4733,sao paulo,SP,...,1306.0,1.0,100.0,20.0,20.0,20.0,eb803377c9315b564bdedad672039306,13289,vinhedo,SP


In [128]:
print("Unique seller cities:", df["seller_city"].nunique())
df["seller_city"].value_counts()

Unique seller cities: 588


seller_city
sao paulo                27156
ibitinga                  7600
curitiba                  2926
santo andre               2885
sao jose do rio preto     2538
                         ...  
campanha                     1
barbacena                    1
goioere                      1
belo horizont                1
avare                        1
Name: count, Length: 588, dtype: int64

In [129]:
# Backtest setup at CITY level: features per (customer_city, customer_state) use ONLY sales before CUTOFF;
# the last 3 months of data are the holdout. 159 city names exist in >1 state, so the key includes the state.
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["item_revenue"] = df["price"] - df["freight_value"]

KEY = ["customer_city", "customer_state"]

# One row per city-order: purchase date + order revenue
city_orders = (
    df.groupby(KEY + ["order_id"], as_index=False)
      .agg(ts=("order_purchase_timestamp", "min"), revenue=("item_revenue", "sum"))
)

HOLDOUT_MONTHS = 3
CUTOFF = city_orders["ts"].max() - pd.DateOffset(months=HOLDOUT_MONTHS)
hist = city_orders[city_orders["ts"] < CUTOFF]
holdout = city_orders[city_orders["ts"] >= CUTOFF]

# Only cities with sales before CUTOFF
city_features = hist.groupby(KEY).agg(last_sale_date=("ts", "max"))
city_features["days_since_last_sale"] = (CUTOFF - city_features["last_sale_date"]).dt.days

for months in (12, 3, 1):
    w = hist[hist["ts"] >= CUTOFF - pd.DateOffset(months=months)].groupby(KEY)
    city_features[f"orders_last_{months}m"] = w.size().reindex(city_features.index, fill_value=0)
    city_features[f"revenue_last_{months}m"] = w["revenue"].sum().reindex(city_features.index, fill_value=0.0)

# Backtest targets: sales in the holdout period
h = holdout.groupby(KEY)
city_features["holdout_orders"] = h.size().reindex(city_features.index, fill_value=0)
city_features["holdout_revenue"] = h["revenue"].sum().reindex(city_features.index, fill_value=0.0)
city_features["sold_in_holdout"] = (city_features["holdout_orders"] > 0).astype(int)

print("cutoff:", CUTOFF, "| cities:", len(city_features), "| positive rate:", city_features["sold_in_holdout"].mean().round(4))
city_features.head(10)


cutoff: 2018-05-29 15:00:37 | cities: 4022 | positive rate: 0.4774


,,last_sale_date,days_since_last_sale,orders_last_12m,revenue_last_12m,orders_last_3m,revenue_last_3m,orders_last_1m,revenue_last_1m,holdout_orders,holdout_revenue,sold_in_holdout
customer_city,customer_state,,,,,,,,,,,
abadia dos dourados,MG,2018-03-02 21:32:44,87,2,282.78,1,178.37,0,0.00,1,21.52,1
abadiania,GO,2018-01-28 14:21:58,121,1,874.46,0,0.00,0,0.00,0,0.00,0
abaete,MG,2018-03-27 11:00:09,63,6,798.89,2,115.66,0,0.00,3,383.79,1
abaetetuba,PA,2018-05-16 18:37:58,12,7,2726.44,4,726.84,1,753.97,3,430.87,1
abaiara,CE,2018-05-11 10:49:39,18,2,193.65,1,137.55,1,137.55,0,0.00,0
abaira,BA,2018-05-01 16:36:01,27,1,-9.07,1,-9.07,1,-9.07,1,122.05,1
abare,BA,2018-05-09 00:44:03,20,2,145.65,1,14.13,1,14.13,0,0.00,0
abatia,PR,2018-04-21 10:58:18,38,3,115.59,1,30.68,0,0.00,0,0.00,0
abdon batista,SC,2017-04-18 10:25:28,406,0,0.00,0,0.00,0,0.00,0,0.00,0


In [131]:
city_features[city_features['orders_last_1m'] >0].sort_values('orders_last_1m', ascending=False).head(10)

,,last_sale_date,days_since_last_sale,orders_last_12m,revenue_last_12m,orders_last_3m,revenue_last_3m,orders_last_1m,revenue_last_1m,holdout_orders,holdout_revenue,sold_in_holdout
customer_city,customer_state,,,,,,,,,,,
sao paulo,SP,2018-05-29 14:54:54,0,9996,1065322.41,3382,385609.82,1179,132941.79,3408,354554.61,1
rio de janeiro,RJ,2018-05-29 14:34:28,0,4525,565290.61,1208,153173.39,409,53247.01,1194,133069.97,1
belo horizonte,MG,2018-05-29 09:40:24,0,1829,188119.45,582,62254.34,187,17267.66,542,63279.93,1
brasilia,DF,2018-05-29 14:21:11,0,1403,163557.88,425,51905.92,136,20350.24,461,51380.84,1
curitiba,PR,2018-05-28 19:29:49,0,1001,123504.51,327,46425.91,96,15209.19,307,32061.54,1
campinas,SP,2018-05-29 13:55:18,0,964,108725.95,314,33068.72,94,8543.21,295,35877.01,1
guarulhos,SP,2018-05-29 11:59:40,0,728,76744.55,242,25009.20,94,9322.42,286,29583.96,1
salvador,BA,2018-05-29 13:50:59,0,828,90539.96,262,29330.32,88,8979.06,227,28254.86,1
porto alegre,RS,2018-05-29 11:10:29,0,934,100788.64,263,28442.38,85,9480.16,233,29797.39,1
